# Protein Language Models Tutorial

This notebook demonstrates how to work with Protein Language Models (pLMs) using Hugging Face Transformers. We'll cover:
- Loading pre-trained pLMs
- Fine-tuning on protein-related tasks
- Evaluating model performance
- Experimenting with different models and tasks

## 1. Install and Import Libraries

First, let's install the necessary libraries and import them.

In [4]:
# Install required packages (run this in a terminal if not already installed)
# !pip install transformers torch datasets evaluate scikit-learn ipywidgets

# Import libraries
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Libraries imported successfully!
PyTorch version: 2.11.0+cpu
CUDA available: False


## 2. Load a Pre-trained Protein Language Model

Protein Language Models are transformer models trained on large protein sequence datasets. We'll use ESM-2, a smaller variant that's good for learning and experimentation.

Popular pLMs include:
- ESM-1b/2 (Meta AI)
- ProtBERT/BERT (Rostlab)
- ProtT5 (Technical University of Munich)

In [8]:
# Model selection widget
model_options = {
    "ESM-2 (8M)": {
        "name": "facebook/esm2_t6_8M_UR50D",
        "year": "2022",
        "publication": "Lin et al., bioRxiv 2022",
        "description": "Evolutionary Scale Modeling 2 - 8M parameters. Trained on UniRef50. Good for small-scale experiments."
    },
    "ESM-2 (35M)": {
        "name": "facebook/esm2_t12_35M_UR50D",
        "year": "2022",
        "publication": "Lin et al., bioRxiv 2022",
        "description": "Evolutionary Scale Modeling 2 - 35M parameters. Better performance than 8M variant."
    },
    "ESM-2 (150M)": {
        "name": "facebook/esm2_t30_150M_UR50D",
        "year": "2022",
        "publication": "Lin et al., bioRxiv 2022",
        "description": "Evolutionary Scale Modeling 2 - 150M parameters. Good balance of performance and speed."
    },
    "ProtBERT": {
        "name": "Rostlab/prot_bert",
        "year": "2019",
        "publication": "Elnaggar et al., IEEE Transactions on Computational Biology and Bioinformatics 2019",
        "description": "BERT model adapted for proteins. Based on BERT-base architecture."
    },
    "ProtT5-XL": {
        "name": "Rostlab/prot_t5_xl_uniref50",
        "year": "2021",
        "publication": "Elnaggar et al., Nature Communications 2021",
        "description": "T5 model for proteins - XL variant. Strong performance on various protein tasks."
    }
}

# Create dropdown widget
model_dropdown = widgets.Dropdown(
    options=list(model_options.keys()),
    value="ESM-2 (8M)",  # Default
    description='Model:',
    style={'description_width': 'initial'}
)

# Output widget for model info
model_info_output = widgets.Output()

def on_model_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected_model = change['new']
        model_data = model_options[selected_model]

        with model_info_output:
            clear_output(wait=True)
            print(f"📊 Model: {selected_model}")
            print(f"🔗 Hugging Face ID: {model_data['name']}")
            print(f"📅 Year: {model_data['year']}")
            print(f"📝 Publication: {model_data['publication']}")
            print(f"📖 Description: {model_data['description']}")

            # Set the global MODEL_NAME
            global MODEL_NAME
            MODEL_NAME = model_data['name']
            print(f"\n✅ MODEL_NAME set to: {MODEL_NAME}")

# Connect the callback
model_dropdown.observe(on_model_change)

# Display widgets
display(model_dropdown)
display(model_info_output)

# Initialize with default model info and set MODEL_NAME
selected_model = model_dropdown.value
model_data = model_options[selected_model]
print(f"📊 Model: {selected_model}")
print(f"🔗 Hugging Face ID: {model_data['name']}")
print(f"📅 Year: {model_data['year']}")
print(f"📝 Publication: {model_data['publication']}")
print(f"📖 Description: {model_data['description']}")

MODEL_NAME = model_data['name']
print(f"\n✅ MODEL_NAME set to: {MODEL_NAME}")

# Update display when model changes
def on_model_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected_model = change['new']
        model_data = model_options[selected_model]

        with model_info_output:
            clear_output(wait=True)
            print(f"📊 Model: {selected_model}")
            print(f"🔗 Hugging Face ID: {model_data['name']}")
            print(f"📅 Year: {model_data['year']}")
            print(f"📝 Publication: {model_data['publication']}")
            print(f"📖 Description: {model_data['description']}")

            # Update the global MODEL_NAME
            global MODEL_NAME
            MODEL_NAME = model_data['name']
            print(f"\n✅ MODEL_NAME updated to: {MODEL_NAME}")

# Connect the callback
model_dropdown.observe(on_model_change)

Dropdown(description='Model:', options=('ESM-2 (8M)', 'ESM-2 (35M)', 'ESM-2 (150M)', 'ProtBERT', 'ProtT5-XL'),…

Output()

📊 Model: ESM-2 (8M)
🔗 Hugging Face ID: facebook/esm2_t6_8M_UR50D
📅 Year: 2022
📝 Publication: Lin et al., bioRxiv 2022
📖 Description: Evolutionary Scale Modeling 2 - 8M parameters. Trained on UniRef50. Good for small-scale experiments.

✅ MODEL_NAME set to: facebook/esm2_t6_8M_UR50D


In [6]:
# Load the selected model
print(f"Loading model: {MODEL_NAME}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model for sequence classification (we'll modify num_labels based on task)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,  # Binary classification (soluble/insoluble)
    problem_type="single_label_classification"
)

print(f"Model loaded successfully!")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model architecture: {model.__class__.__name__}")

Loading model: facebook/esm2_t6_8M_UR50D


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 13843.48it/s]
[transformers] EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!
Number of parameters: 7,512,443
Model architecture: EsmForSequenceClassification


## 3. Prepare Protein Sequence Data

We'll create a synthetic protein dataset for demonstration purposes, simulating protein sequences labeled as soluble or insoluble. This allows us to demonstrate the pLM fine-tuning process without requiring access to external datasets.

**Dataset Details**: 
- **Synthetic Data**: Randomly generated protein sequences with binary solubility labels
- **Training set**: 500 samples
- **Test set**: 100 samples
- **Task**: Binary classification (soluble vs insoluble)

**Note**: You can try loading real datasets by setting `loading_external_dataset = True` in the code cell below. In a real scenario, you would use actual experimental data such as:
- DeepSol dataset (when available on Hugging Face)
- Secondary structure prediction datasets
- Protein function annotation datasets
- Stability prediction datasets

For real protein data, you can explore:
- UniProt database
- Protein Data Bank (PDB)
- Various datasets on Hugging Face Hub

In [15]:
# Set this flag to True to try loading external datasets, False to use synthetic data
loading_external_dataset = False

if loading_external_dataset:
    # Try to load the DeepSol dataset from Hugging Face
    print("Attempting to load DeepSol dataset from Hugging Face...")
    try:
        dataset = load_dataset("proteinea/solubility")
        dataset = dataset.rename_columns({"sequences": "sequence", "labels": "label"})
        dataset = dataset.map(lambda b: {"label": [int(x) for x in b["label"]]}, batched=True)
        dataset = dataset.shuffle(seed=42)
        print("✓ Successfully loaded DeepSol dataset!")
        
        # For demo purposes, use a small subset
        train_dataset = dataset['train'].select(range(500))  # 500 training samples
        test_dataset = dataset['test'].select(range(100))   # 100 test samples
        
    except Exception as e:
        print(f"✗ Failed to load DeepSol dataset: {e}")
        print("Falling back to synthetic data...")
        loading_external_dataset = False

if not loading_external_dataset:
    # Create a synthetic protein dataset for demonstration
    print("Creating synthetic protein dataset for demonstration...")

    # Generate synthetic protein sequences and labels
    import random

    def generate_protein_sequence(length=100):
        """Generate a random protein sequence."""
        amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
        return ''.join(random.choice(amino_acids) for _ in range(length))

    # Create synthetic data
    n_train = 500
    n_test = 100

    train_sequences = [generate_protein_sequence(random.randint(50, 150)) for _ in range(n_train)]
    train_labels = [random.choice([0, 1]) for _ in range(n_train)]  # 0: insoluble, 1: soluble

    test_sequences = [generate_protein_sequence(random.randint(50, 150)) for _ in range(n_test)]
    test_labels = [random.choice([0, 1]) for _ in range(n_test)]

    # Create datasets using datasets library
    from datasets import Dataset, DatasetDict

    train_dataset = Dataset.from_dict({
        'sequence': train_sequences,
        'label': train_labels
    })

    test_dataset = Dataset.from_dict({
        'sequence': test_sequences,
        'label': test_labels
    })

    dataset = DatasetDict({
        'train': train_dataset,
        'test': test_dataset
    })

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Check the structure
print("\nDataset structure:")
print(train_dataset[0])

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["sequence"],
        truncation=True,
        padding=False,  # We'll use dynamic padding
        max_length=512
    )

# Tokenize datasets
print("\nTokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["sequence"])
tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=["sequence"])

print("Data preparation complete!")

Creating synthetic protein dataset for demonstration...
Training samples: 500
Test samples: 100

Dataset structure:
{'sequence': 'EAKIIFEVDWQCADHITYAVHVQIRWKAGQMKFHMEDPENNYKCRVEPDVLYNWHDCILDIEPKRNGNNHKDYGVIGRPKVIMCICMPKDHWMHSPRFKFIVVKWQWPNIFTSDCEFGQYDPPYRTKVAEV', 'label': 0}

Tokenizing datasets...


Map: 100%|██████████| 100/100 [00:00<00:00, 3226.74 examples/s]

Data preparation complete!


## 4. Fine-tune the Model on a Task

Fine-tuning adapts the pre-trained pLM to our specific task (solubility prediction). This process updates the model's weights using our labeled data.

The process involves:
1. Setting up training arguments (learning rate, batch size, epochs)
2. Creating a Trainer object
3. Training the model
4. Saving the fine-tuned model

In [13]:
# Define metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,  # Small batch size for demo
    per_device_eval_batch_size=4,
    num_train_epochs=2,  # Few epochs for demo
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=50,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
print("Starting fine-tuning...")
trainer.train()

# Save the model
trainer.save_model("./fine_tuned_model")
print("Model saved to ./fine_tuned_model")

Starting fine-tuning...


c:\Users\soura\Code\2026\plm-starter\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.694923,0.690549,0.560000,0.578947,0.932203,0.714286
2,0.686506,0.694715,0.490000,0.576923,0.508475,0.540541


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 56.98it/s]
c:\Users\soura\Code\2026\plm-starter\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 79.82it/s]

Model saved to ./fine_tuned_model


## 5. Evaluate Model Performance

After fine-tuning, we evaluate the model's performance on the test set using various metrics. This helps us understand how well the model generalizes to unseen data.

In [14]:
# Evaluate the model
print("Evaluating model on test set...")
eval_results = trainer.evaluate()

print("\nEvaluation Results:")
for metric, value in eval_results.items():
    print(f"{metric}: {value:.4f}")

# You can also make predictions on individual examples
print("\nExample predictions:")
predictions = trainer.predict(tokenized_test)
pred_labels = np.argmax(predictions.predictions, axis=1)

# Show a few examples
for i in range(5):
    true_label = test_dataset[i]['label']
    pred_label = pred_labels[i]
    sequence = test_dataset[i]['sequence'][:50] + "..."
    status = "✓" if true_label == pred_label else "✗"
    print(f"{status} True: {true_label}, Pred: {pred_label} | {sequence}")

Evaluating model on test set...


c:\Users\soura\Code\2026\plm-starter\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.686506,0.690549,2,0.560000,0.578947,0.932203,0.714286



Evaluation Results:
eval_loss: 0.6905
eval_accuracy: 0.5600
eval_precision: 0.5789
eval_recall: 0.9322
eval_f1: 0.7143

Example predictions:


c:\Users\soura\Code\2026\plm-starter\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


✓ True: 1, Pred: 1 | TPNPKEAIFMRRDPKMVPHTIQDVTGIWYSEDKWYRIFIGCIWGAFNYWK...
✓ True: 1, Pred: 1 | YVLGTETVYGNMKGDGHSMIRNTSGQYKQFPAPWWIHLQDKYTKFSQLWR...
✗ True: 0, Pred: 1 | PWMSEQGIGMMEPMGRIWGVDGRAKMEHWMNSRSQHQQLANVGWVCLQLD...
✗ True: 0, Pred: 1 | YQIKAARPFWCAYFKWFQLSWDRDMTLASCPKHRQLVFQKCSYHAYMRYW...
✓ True: 1, Pred: 1 | DWHYKHNWNNRAWMQTKAYGMIINFQFGYQWGRAKQDRGWAYDSTQHNYH...


## 6. Test with Different Models and Tasks

To experiment with other models and tasks:

### Different Models:
- `"Rostlab/prot_bert"` - ProtBERT model
- `"facebook/esm2_t12_35M_UR50D"` - Larger ESM-2 model
- `"facebook/esm1v_t33_650M_UR90S_1"` - ESM-1v for variant effect prediction

### Different Tasks:
- **Secondary Structure Prediction**: Use token classification instead of sequence classification
- **Protein Function Prediction**: Multi-label classification
- **Stability Prediction**: Regression task

### Example: Try a Different Model
Uncomment and modify the code below to try different models.

In [ ]:
# Example: Try a different model
# Uncomment the lines below to try ProtBERT instead

# MODEL_NAME = "Rostlab/prot_bert"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=2,
#     problem_type="single_label_classification"
# )
# print(f"Switched to model: {MODEL_NAME}")

# For regression tasks (e.g., stability prediction), change to:
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=1,  # Single output for regression
#     problem_type="regression"
# )

# For token classification (e.g., secondary structure):
# from transformers import AutoModelForTokenClassification
# model = AutoModelForTokenClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=3  # H, E, C for helix, sheet, coil
# )

print("Experiment with different models by changing MODEL_NAME and adjusting the model type!")
print("\nNext steps:")
print("1. Try different pre-trained models using the dropdown widget")
print("2. Set loading_external_dataset = True to try loading real protein datasets")
print("3. Experiment with different datasets/tasks")
print("4. Adjust hyperparameters (learning rate, batch size, epochs)")
print("5. Use the model for inference on new sequences")